In [ ]:
# === Cell 1: Data Root ===
import os

# Data root; override with the KI_SORT_ROOT environment variable.
KI_SORT_ROOT = os.environ.get("KI_SORT_ROOT", "/VOL_D/fio.h")


In [ ]:
# === Cell 2: Prerequisites ===
"""
Runs in the `ir-class` kernel (or any environment with pandas, numpy, opencv and
matplotlib) up to the Per-Object Perspective Spread cell. The evaluation-tier
cells from Evaluation-Tier Segmentation Setup onward need the `yolo-train`
kernel (torch, ultralytics).

Scope of the GT part:
  * GT segmentation masks and GT material classes only; no inference.
  * Analyses:
      1. EN 18120 vs RecyClass severity lookup: interval width, severity only.
      2. Thickness correction factor levels (6 variants): effect on I.
      3. Per-object index spread across perspectives.
      4. Region size (mm^2) reference table.
  * GT segmentation exists only for the 40 Initial-dataset objects. The 31
    Extended-dataset objects have thickness/weight measurements but no GT mask;
    they are excluded from every per-perspective computation and from every
    aggregated factor (per-type, per-material-pair, global).
"""


In [ ]:
# === Cell 3: Imports ===

%matplotlib inline

import re
import sys
import platform
import warnings
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from IPython.display import display

warnings.filterwarnings("ignore", category=UserWarning)


In [ ]:
# === Cell 4: Config ===

PROJECT_NAME = "20260821_Impurity_Index_Computation"

# --- Paths ---
PROJECT_ROOT = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/07_Impurity_Index")
RESULTS_DIR  = PROJECT_ROOT / "results"

SEVERITY_LOOKUP_EN_CSV = PROJECT_ROOT / "severity_lookup_EN18120.csv"
SEVERITY_LOOKUP_RC_CSV = PROJECT_ROOT / "severity_lookup_RecyClass.csv"

# Thickness/weight measurement log (per-object reference measurements).
THICKNESS_WEIGHT_CSV = PROJECT_ROOT / "20260820_Thickness_Weight_Measurements.csv"

# --- Ground truth ---
GT_SEG_LABELS_DIR   = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/labels/all")
IR_GROUND_TRUTH_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_NIR_Dataset/spectra_ground_truth_v8.csv")
IMAGES_DIR          = Path(f"{KI_SORT_ROOT}/KI_Sort_Datasets/Combined_Image_Dataset/1080_resolution")

# --- Rasterisation canvas ---
# Fallback size, used only when the source image is missing. Coverage ratios
# (and therefore w_r) are scale-invariant.
IMG_W_FALLBACK = 1920
IMG_H_FALLBACK = 1080

# --- Pixel -> physical calibration ---
# Same optical setup as the measurement point sampling notebook. mm^2 values are
# projected areas at the calibration plane; only the Region Size Reference Table
# reports them, no ratio or index calculation uses absolute areas.
PIXELS_PER_MM = 3.55
MM2_PER_PX2   = 1.0 / (PIXELS_PER_MM ** 2)

# --- Dataset semantics ---
CLASS_MAP = {"body": 0, "cap": 1, "label": 2}
ID_TO_CLASS = {v: k for k, v in CLASS_MAP.items()}
ATTACHMENT_COMPONENTS = ("label", "cap")
# Both severity lookup CSVs key their component column as "label" / "cap".

MATERIAL_ALIASES = {"HDPE": "PE", "LDPE": "PE"}
SUPPORTED_BODY_MATERIALS = ("PET", "PE", "PP", "PS")
MATERIAL_ORDER = ["PE", "PET", "PP", "PS"]

RANDOM_SEED = 42

# Plot toggle: veto_strict markers (open black circles) on the Index by Body
# Material plots.
SHOW_VETO_MARKERS = False

# --- Output ---
ALLOW_OVERWRITE = True   # False protects existing result CSVs from being overwritten

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

IMAGE_REGEX = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_per_(?P<per>\d+)"
)

print(f"PROJECT_NAME: {PROJECT_NAME}")
print(f"Scope:        static GT only, Initial-dataset objects only (see Cell 2)")


In [ ]:
# === Cell 5: Environment and Path Checks ===

print("Environment")
print("-" * 78)
print(f"Python:    {sys.version.split()[0]}")
print(f"Platform:  {platform.platform()}")
print(f"NumPy:     {np.__version__}")
print(f"pandas:    {pd.__version__}")
print(f"OpenCV:    {cv2.__version__}")

PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def _check(label, path, kind="dir", required=True):
    ok  = path.is_dir() if kind == "dir" else path.is_file()
    tag = "DIR " if kind == "dir" else "FILE"
    note = "" if required else "  (optional)"
    print(f"  [{tag}] {'OK' if ok else 'MISSING':8s} {label}: {path}{note}")
    return ok or not required


print()
print("Path checks")
print("-" * 78)
all_ok = True
all_ok &= _check("RESULTS_DIR",             RESULTS_DIR,             "dir")
all_ok &= _check("SEVERITY_LOOKUP_EN_CSV",  SEVERITY_LOOKUP_EN_CSV,  "file")
all_ok &= _check("SEVERITY_LOOKUP_RC_CSV",  SEVERITY_LOOKUP_RC_CSV,  "file")
all_ok &= _check("THICKNESS_WEIGHT_CSV",    THICKNESS_WEIGHT_CSV,   "file")
all_ok &= _check("GT_SEG_LABELS_DIR",       GT_SEG_LABELS_DIR,      "dir")
all_ok &= _check("IR_GROUND_TRUTH_CSV",     IR_GROUND_TRUTH_CSV,    "file")
all_ok &= _check("IMAGES_DIR",              IMAGES_DIR,             "dir", required=False)

print()
print("Config summary")
print("-" * 78)
print(f"  PIXELS_PER_MM:   {PIXELS_PER_MM}  ->  MM2_PER_PX2 = {MM2_PER_PX2:.6f}")
print(f"  CLASS_MAP:       {CLASS_MAP}")
print(f"  Components:      {ATTACHMENT_COMPONENTS}  (cap -> EN 'rigid closure')")
print(f"  ALLOW_OVERWRITE: {ALLOW_OVERWRITE}")

print()
if not all_ok:
    raise FileNotFoundError(
        "One or more required paths are missing -- see [MISSING] lines above. "
        "Fix Cell 4 before continuing; a missing GT source silently breaks material "
        "resolution (Cell 9) and index_resolvable (Cell 11) many cells downstream "
        "rather than failing here where the actual cause is visible."
    )
print("All paths OK.")


In [ ]:
# === Cell 6: Loaders and Asset Index ===
# Parses YOLO GT label files and indexes images and GT materials per object.

_STEM_RE = re.compile(
    r"^(?P<material>[A-Za-z0-9]+)_obj_(?P<object_id>\d+)_per_(?P<perspective_id>\d+)"
)


def normalise_material(material: str) -> str:
    """Filename / GT-CSV material token -> polymer name used by the lookup."""
    if material is None:
        return None
    m = str(material).strip()
    if m == "" or m.lower() in ("nan", "none"):
        return None
    return MATERIAL_ALIASES.get(m.upper(), m)


def parse_stem(stem: str):
    """'HDPE_obj_01_per_03_2026-04-21' -> ('HDPE', 1, 3, 'HDPE_obj_01')."""
    m = _STEM_RE.match(stem)
    if not m:
        return None
    material  = m.group("material")
    object_id = int(m.group("object_id"))
    per_id    = int(m.group("perspective_id"))
    object_name = f"{material}_obj_{m.group('object_id')}"
    return material, object_id, per_id, object_name


def parse_label_file(txt_path: Path) -> list:
    """Parse one YOLO segmentation .txt (normalised polygons); an empty file
    yields an empty instance list."""
    instances = []
    if not txt_path.is_file():
        return instances
    with txt_path.open("r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            cls_id = int(float(parts[0]))
            coords = [float(v) for v in parts[1:]]
            if len(coords) % 2 != 0:
                coords = coords[:-1]
            instances.append({
                "class_id":     cls_id,
                "polygon_norm": list(zip(coords[0::2], coords[1::2])),
            })
    return instances


def resolve_image_size(image_path):
    """True (H, W) of the source image, or the configured fallback canvas."""
    if image_path is not None and Path(image_path).is_file():
        img = cv2.imread(str(image_path))
        if img is not None:
            return img.shape[0], img.shape[1]
    return IMG_H_FALLBACK, IMG_W_FALLBACK


def px2_to_mm2(area_px2) -> float:
    """Convert an area in px^2 to mm^2; None/NaN -> NaN."""
    if area_px2 is None or (isinstance(area_px2, float) and np.isnan(area_px2)):
        return np.nan
    return float(area_px2) * MM2_PER_PX2


# --- Image pool: image_index[object_name][perspective_id] = Path ---
# Used only to read the true canvas size; GT masks are read from .txt files.
image_index = defaultdict(dict)
if IMAGES_DIR.is_dir():
    for _p in sorted(IMAGES_DIR.iterdir()):
        if not (_p.is_file() and _p.suffix.lower() in IMAGE_EXTS):
            continue
        _parsed = parse_stem(_p.stem)
        if _parsed is None:
            continue
        _, _, _per, _obj = _parsed
        image_index[_obj][_per] = _p
image_index = {k: dict(sorted(v.items())) for k, v in sorted(image_index.items())}
_n_images = sum(len(v) for v in image_index.values())
print(f"Images indexed:   {_n_images} across {len(image_index)} object(s)")

# --- GT materials (semicolon-delimited, UTF-8-BOM) ---
# Single material source for body/label/cap. The thickness/weight CSV joins onto
# it by object_name only; its own material columns are not used here.
gt_materials = {}
if IR_GROUND_TRUTH_CSV.is_file():
    _gt = pd.read_csv(IR_GROUND_TRUTH_CSV, sep=";", encoding="utf-8-sig")
    _gt.columns = [c.strip() for c in _gt.columns]
    for _, _r in _gt.iterrows():
        _key = str(_r["filename"]).strip()
        if not _key or _key.lower() == "nan":
            continue
        gt_materials[_key] = {
            "body":  normalise_material(_r.get("body_class")),
            "label": normalise_material(_r.get("label_class")),
            "cap":   normalise_material(_r.get("cap_class")),
        }
print(f"GT material rows: {len(gt_materials)} object(s)")
if len(gt_materials) == 0:
    raise RuntimeError(
        f"gt_materials is EMPTY -- IR_GROUND_TRUTH_CSV ({IR_GROUND_TRUTH_CSV}) did not "
        "load any rows. Every perspective's body/label/cap material will resolve to "
        "None and index_resolvable will be False almost everywhere -- check the path "
        "in Cell 4 before continuing."
    )


In [ ]:
# === Cell 7: Severity Lookup Tables ===
# Loads the EN 18120 and RecyClass lookup tables (same schema: body_material,
# component, attachment_material, severity_lo, severity_hi, source_table, note).
# Index and factor computations use EN 18120 only; RecyClass is used only in the
# Severity Interval Width Comparison.

_SEV_REQUIRED_COLS = [
    "body_material", "component", "attachment_material",
    "severity_lo", "severity_hi",
]


def _load_severity_table(path: Path, name: str):
    """Load and validate one severity table; return (DataFrame, lookup dict)."""
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]

    _missing = [c for c in _SEV_REQUIRED_COLS if c not in df.columns]
    if _missing:
        raise ValueError(f"{name} is missing column(s): {_missing}")

    for _c in ("body_material", "component", "attachment_material"):
        df[_c] = df[_c].astype(str).str.strip()
    for _c in ("severity_lo", "severity_hi"):
        df[_c] = pd.to_numeric(df[_c], errors="raise")

    _bad_order = df[df["severity_lo"] > df["severity_hi"]]
    if len(_bad_order):
        raise ValueError(f"{name}: severity_lo > severity_hi in {len(_bad_order)} row(s):\n{_bad_order}")

    _bad_range = df[(df["severity_lo"] < 0) | (df["severity_hi"] > 1)]
    if len(_bad_range):
        raise ValueError(f"{name}: severity outside [0, 1] in {len(_bad_range)} row(s):\n{_bad_range}")

    _dupes = df.duplicated(subset=["body_material", "component", "attachment_material"], keep=False)
    if _dupes.any():
        raise ValueError(f"{name}: duplicate lookup key(s):\n{df[_dupes]}")

    _bad_component = set(df["component"]) - set(ATTACHMENT_COMPONENTS)
    if _bad_component:
        raise ValueError(f"{name}: unknown component(s) {_bad_component}; expected only {ATTACHMENT_COMPONENTS}")

    lookup = {
        (r["body_material"], r["component"], r["attachment_material"]):
            (float(r["severity_lo"]), float(r["severity_hi"]))
        for _, r in df.iterrows()
    }
    return df, lookup


severity_df_EN, SEVERITY_EN = _load_severity_table(SEVERITY_LOOKUP_EN_CSV, "severity_lookup_EN18120.csv")
severity_df_RC, SEVERITY_RC = _load_severity_table(SEVERITY_LOOKUP_RC_CSV, "severity_lookup_RecyClass.csv")

for _name, _df in [("EN18120", severity_df_EN), ("RecyClass", severity_df_RC)]:
    _n_interval = int((_df["severity_lo"] != _df["severity_hi"]).sum())
    print(f"{_name}: {len(_df)} rows, {_n_interval} interval ({_n_interval/len(_df):.0%})")

_merged = severity_df_EN.merge(
    severity_df_RC, on=["body_material", "component", "attachment_material"], suffixes=("_en", "_rc")
)
_diverge = _merged[
    (_merged.severity_lo_en != _merged.severity_lo_rc) | (_merged.severity_hi_en != _merged.severity_hi_rc)
]
print(f"Rows where EN18120 and RecyClass severity differ: {len(_diverge)} / {len(_merged)}")
display(_diverge[["body_material", "component", "attachment_material",
                   "severity_lo_en", "severity_hi_en", "severity_lo_rc", "severity_hi_rc"]])

# Status vocabulary for a single (component) severity resolution.
SEV_STATUS_OK           = "ok"
SEV_STATUS_ABSENT       = "absent"
SEV_STATUS_UNCLASSIFIED = "unclassified"
SEV_STATUS_BODY_ABSENT  = "body_absent"
SEV_STATUS_UNSUPPORTED  = "unsupported_body"
SEV_STATUS_MISSING      = "missing_lookup"


def resolve_severity(body_material, component, attachment_material, region_present, table):
    """Return (severity_lo, severity_hi, status) for one component of one
    perspective, using the given lookup dict (SEVERITY_EN or SEVERITY_RC)."""
    if not region_present:
        return 0.0, 0.0, SEV_STATUS_ABSENT
    if attachment_material is None:
        return np.nan, np.nan, SEV_STATUS_UNCLASSIFIED
    if body_material is None:
        return np.nan, np.nan, SEV_STATUS_BODY_ABSENT
    if body_material not in SUPPORTED_BODY_MATERIALS:
        return np.nan, np.nan, SEV_STATUS_UNSUPPORTED
    hit = table.get((body_material, component, attachment_material))
    if hit is None:
        return np.nan, np.nan, SEV_STATUS_MISSING
    return hit[0], hit[1], SEV_STATUS_OK


In [ ]:
# === Cell 8: Coverage Computation ===
#   A_total = area( body_union OR label_union OR cap_union )
#   w_label = A_label / A_total ;  w_cap = A_cap / A_total
# Areas come from a boolean mask union, not summed polygon areas; the label/cap
# overlap is reported separately and not clipped.

def rasterise_polygon(poly_norm, h, w) -> np.ndarray:
    """Rasterise one normalised polygon to a boolean (h, w) mask."""
    if poly_norm is None or len(poly_norm) < 3:
        return np.zeros((h, w), dtype=bool)
    pts = np.array(
        [(int(round(x * w)), int(round(y * h))) for x, y in poly_norm],
        dtype=np.int32,
    ).reshape(-1, 1, 2)
    canvas = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(canvas, [pts], 1)
    return canvas.astype(bool)


def union_mask(instances, class_name, h, w) -> np.ndarray:
    """Boolean union of all instance masks of one class."""
    cls_id = CLASS_MAP[class_name]
    mask = np.zeros((h, w), dtype=bool)
    for inst in instances:
        if int(inst["class_id"]) == cls_id:
            mask |= rasterise_polygon(inst["polygon_norm"], h, w)
    return mask


def compute_coverage(instances, h, w) -> dict:
    """Region areas (px^2, mm^2), coverage weights w_label/w_cap and presence flags."""
    body  = union_mask(instances, "body",  h, w)
    label = union_mask(instances, "label", h, w)
    cap   = union_mask(instances, "cap",   h, w)

    total = body | label | cap
    body_exposed = body & ~(label | cap)

    a_body    = int(body.sum())
    a_label   = int(label.sum())
    a_cap     = int(cap.sum())
    a_exposed = int(body_exposed.sum())
    a_total   = int(total.sum())

    ov_label_cap = int((label & cap).sum())

    n_body  = sum(1 for i in instances if int(i["class_id"]) == CLASS_MAP["body"])
    n_cap   = sum(1 for i in instances if int(i["class_id"]) == CLASS_MAP["cap"])
    n_label = sum(1 for i in instances if int(i["class_id"]) == CLASS_MAP["label"])

    undefined = a_total == 0
    w_label = np.nan if undefined else a_label / a_total
    w_cap   = np.nan if undefined else a_cap   / a_total
    w_sum   = np.nan if undefined else w_label + w_cap

    return {
        "n_body_instances":  n_body,
        "n_cap_instances":   n_cap,
        "n_label_instances": n_label,
        "label_present":     a_label > 0,
        "cap_present":       a_cap   > 0,
        "body_present":      a_body  > 0,

        "body_area_px2":          a_body,
        "label_area_px2":         a_label,
        "cap_area_px2":           a_cap,
        "total_surface_area_px2": a_total,

        "body_area_mm2":          px2_to_mm2(a_body),
        "label_area_mm2":         px2_to_mm2(a_label),
        "cap_area_mm2":           px2_to_mm2(a_cap),
        "total_surface_area_mm2": px2_to_mm2(a_total),

        "w_label":     w_label,
        "w_cap":       w_cap,
        "weights_sum": w_sum,
        "weights_sum_exceeds_one": bool(w_sum > 1.0 + 1e-9) if not undefined else False,

        "overlap_label_cap_area_px2": ov_label_cap,
        "undefined": undefined,
    }


In [ ]:
# === Cell 9: Material Resolution ===
# Materials come from gt_materials (Loaders and Asset Index cell). A region absent
# from the segmentation gets no material and status "absent" (weight 0, severity 0),
# e.g. lid-type caps annotated as part of the body.

def resolve_materials(object_name, coverage):
    """Materials for body / label / cap on one perspective, from GT."""
    present = {
        "body":  coverage["body_present"],
        "label": coverage["label_present"],
        "cap":   coverage["cap_present"],
    }
    out = {}
    for region in ("body", "label", "cap"):
        if not present[region]:
            out[region] = {"material": None, "status": "absent"}
            continue
        mat = gt_materials.get(object_name, {}).get(region)
        out[region] = {"material": mat, "status": "ok" if mat else "no_gt_entry"}
    return out


In [ ]:
# === Cell 10: Impurity Index Computation ===
#   I = SUM_r  w_r * s_r         over r in {label, cap}, EN 18120 severity only.
#   I_lo = SUM_r w_r * s_r_lo ;  I_hi = SUM_r w_r * s_r_hi   (bounds per region)
# RecyClass severity bounds are stored per region for the severity width
# comparison but never enter I_lo/I_hi. veto_strict: s_lo = s_hi = 1;
# veto_loose: s_hi = 1 (both EN 18120).

WEIGHT_OF  = {"label": "w_label", "cap": "w_cap"}
PRESENT_OF = {"label": "label_present", "cap": "cap_present"}


def compute_index(coverage, materials) -> dict:
    """Impurity index (I_lo, I_hi, veto flags) for one perspective, EN 18120.
    Also attaches RecyClass severity bounds per region (not used in I_lo/I_hi)."""
    body_material = materials["body"]["material"]

    out = {
        "body_material":        body_material,
        "body_material_status": materials["body"]["status"],
    }

    i_lo = 0.0
    i_hi = 0.0
    veto_strict = False
    veto_loose  = False
    resolvable  = True
    reasons     = []

    for region in ATTACHMENT_COMPONENTS:
        present = coverage[PRESENT_OF[region]]
        weight  = coverage[WEIGHT_OF[region]]
        mat     = materials[region]["material"]

        s_lo, s_hi, status = resolve_severity(body_material, region, mat, present, SEVERITY_EN)
        s_lo_rc, s_hi_rc, status_rc = resolve_severity(body_material, region, mat, present, SEVERITY_RC)

        out[f"{region}_material"]        = mat
        out[f"{region}_material_status"] = materials[region]["status"]
        out[f"{region}_severity_lo"]     = s_lo
        out[f"{region}_severity_hi"]     = s_hi
        out[f"{region}_severity_status"] = status
        out[f"{region}_severity_lo_rc"]  = s_lo_rc
        out[f"{region}_severity_hi_rc"]  = s_hi_rc
        out[f"{region}_severity_status_rc"] = status_rc
        out[f"{region}_contribution_lo"] = (
            np.nan if (np.isnan(s_lo) or pd.isna(weight)) else weight * s_lo
        )
        out[f"{region}_contribution_hi"] = (
            np.nan if (np.isnan(s_hi) or pd.isna(weight)) else weight * s_hi
        )

        if status == SEV_STATUS_ABSENT:
            continue
        if np.isnan(s_lo) or pd.isna(weight):
            resolvable = False
            reasons.append(f"{region}:{status}")
            continue

        i_lo += weight * s_lo
        i_hi += weight * s_hi
        if s_lo == 1.0 and s_hi == 1.0:
            veto_strict = True
        if s_hi == 1.0:
            veto_loose = True

    if coverage["undefined"]:
        resolvable = False
        reasons.append("no_segmentation")

    out["I_lo"] = i_lo if resolvable else np.nan
    out["I_hi"] = i_hi if resolvable else np.nan
    out["I_width"] = (i_hi - i_lo) if resolvable else np.nan
    out["veto_strict"] = int(veto_strict) if resolvable else np.nan
    out["veto_loose"]  = int(veto_loose)  if resolvable else np.nan
    out["index_resolvable"] = resolvable
    out["index_unresolved_reason"] = ";".join(reasons) if reasons else ""
    return out


def evaluate_perspective(object_name, material_token, object_id, perspective_id, stem) -> dict:
    """Full chain for one GT perspective: masks -> coverage -> materials -> index."""
    image_path = image_index.get(object_name, {}).get(perspective_id)
    h, w = resolve_image_size(image_path)

    instances = parse_label_file(GT_SEG_LABELS_DIR / f"{stem}.txt")
    coverage  = compute_coverage(instances, h, w)
    materials = resolve_materials(object_name, coverage)
    index     = compute_index(coverage, materials)

    return {
        "mask_source":    "gt",
        "stem":           stem,
        "object_name":    object_name,
        "material_token": material_token,
        "object_id":      object_id,
        "perspective_id": perspective_id,
        "canvas_h": h, "canvas_w": w,
        **coverage,
        **index,
    }


In [ ]:
# === Cell 11: Batch Evaluation ===
# Evaluates every GT perspective; one row per perspective, no aggregation.

def build_work_list():
    """(object_name, material, object_id, perspective_id, stem) for every GT label file."""
    work = []
    for txt in sorted(GT_SEG_LABELS_DIR.glob("*.txt")):
        parsed = parse_stem(txt.stem)
        if parsed is None:
            continue
        material, object_id, per_id, object_name = parsed
        work.append((object_name, material, object_id, per_id, txt.stem))
    return sorted(work, key=lambda t: (t[0], t[3]))


_work = build_work_list()
print(f"Evaluating {len(_work)} GT perspective(s) ...")

_t0 = datetime.now()
_rows = []
for _i, (_obj, _mat, _oid, _per, _stem) in enumerate(_work, start=1):
    _rows.append(evaluate_perspective(_obj, _mat, _oid, _per, _stem))
    if _i % 50 == 0 or _i == len(_work):
        print(f"  {_i}/{len(_work)}")

df_perspective = pd.DataFrame(_rows)
_elapsed = (datetime.now() - _t0).total_seconds()
print(f"Done in {_elapsed:.1f}s -- {len(df_perspective)} rows, "
      f"{df_perspective['object_name'].nunique()} object(s)")

_n_resolvable = int(df_perspective["index_resolvable"].sum())
print(f"index_resolvable: {_n_resolvable} / {len(df_perspective)} "
      f"({_n_resolvable/len(df_perspective):.0%})")

_body_materials_seen = sorted(df_perspective["body_material"].dropna().unique())
print(f"Body materials seen: {_body_materials_seen}")
_unsupported = set(_body_materials_seen) - set(SUPPORTED_BODY_MATERIALS)
if _unsupported:
    print(f"  WARNING -- unsupported body material(s) present: {_unsupported}")


In [ ]:
# === Cell 12: Per-Perspective CSV Export ===

_out_path = RESULTS_DIR / f"{PROJECT_NAME}_gt_index_per_perspective.csv"
if _out_path.is_file() and not ALLOW_OVERWRITE:
    raise RuntimeError(f"{_out_path} already exists and ALLOW_OVERWRITE=False.")

df_perspective.to_csv(_out_path, index=False)
print(f"Saved: {_out_path}  ({len(df_perspective)} rows)")


In [ ]:
# === Cell 13: Thickness Correction Factors ===
#   f_r = t_r / t_body      r in {label, cap}      (object-level thickness ratio)
#
# Five factor levels; aggregated levels carry an overbar in the LaTeX notation.
#   code (column/constant)         level             LaTeX
#   f_label_obj,  f_cap_obj        per-object         f_r^{object}
#   f_label_type, f_cap_type       per-type           \bar f_r^{type}
#   f_label_pair, f_cap_pair       per-material-pair   \bar f_r^{pair}
#   F_LABEL_POOLED, F_CAP_POOLED   global, pooled      \bar f_r^{pool}
#   F_LABEL_BALANCED, F_CAP_BALANCED  global, balanced \bar f_r^{bal}
#
# Aggregated factors (type, pair, pool, bal) use Initial-dataset rows only, so
# Extended-dataset objects do not shift the group means applied to Initial objects.
# Printed labels (label_type == "printed") have no label thickness, so f_label^{object}
# is NaN and the Apply Correction-Factor Variants cell uses factor 1. Under EN 18120
# their same-material severity is (0, 0) (body PE/PP/PS), so I is unaffected.

_tw = pd.read_csv(THICKNESS_WEIGHT_CSV, sep=";", decimal=",")
_tw.columns = [c.strip() for c in _tw.columns]
print(f"Thickness/weight measurements: {len(_tw)} objects "
      f"({(_tw.dataset == 'Initial').sum()} Initial, {(_tw.dataset == 'Extended').sum()} Extended)")

_tw["f_label_obj"] = _tw["label_thickness_mean_mm"] / _tw["body_thickness_mean_mm"]
_tw["f_cap_obj"]   = _tw["cap_thickness_mean_mm"]   / _tw["body_thickness_mean_mm"]

# Lid-type caps are annotated as part of the body and have no measurement;
# excluded explicitly.
_tw_cap = _tw[_tw["cap_type"] != "lid"].copy()

_tw_init     = _tw[_tw["dataset"] == "Initial"].copy()
_tw_cap_init = _tw_cap[_tw_cap["dataset"] == "Initial"].copy()

# --- f_r^{object}: the object's own measurement ---
factors_per_object = _tw[["object_name", "dataset", "body_class", "body_type",
                          "label_class", "label_type", "cap_class", "cap_type",
                          "f_label_obj", "f_cap_obj"]].copy()

# --- \bar f_r^{type}: mean by label_type / cap_type ---
# Grouped by the type column only, regardless of material.
_lab_valid_init = _tw_init[_tw_init["f_label_obj"].notna()]
label_type_means = _lab_valid_init.groupby("label_type")["f_label_obj"].mean()

_cap_valid_init = _tw_cap_init[_tw_cap_init["f_cap_obj"].notna()]
cap_type_means = _cap_valid_init.groupby("cap_type")["f_cap_obj"].mean()

print()
print("f_label^{type} by label_type (Initial-only):")
display(pd.concat([label_type_means.rename("mean"),
                    _lab_valid_init.groupby("label_type")["f_label_obj"].std().rename("std"),
                    _lab_valid_init["label_type"].value_counts().rename("n")], axis=1))

print("f_cap^{type} by cap_type (Initial-only, lid excluded):")
display(pd.concat([cap_type_means.rename("mean"),
                    _cap_valid_init.groupby("cap_type")["f_cap_obj"].std().rename("std"),
                    _cap_valid_init["cap_type"].value_counts().rename("n")], axis=1))

factors_per_object["f_label_type"] = factors_per_object["label_type"].map(label_type_means)
factors_per_object["f_cap_type"] = factors_per_object["cap_type"].map(cap_type_means)

# --- \bar f_r^{pair}: body_class x label_class / body_class x cap_class ---
pair_label_means = _lab_valid_init.groupby(["body_class", "label_class"])["f_label_obj"].mean()
pair_cap_means = _cap_valid_init.groupby(["body_class", "cap_class"])["f_cap_obj"].mean()

print()
print("f_label^{pair} by (body_class, label_class) (Initial-only):")
display(pd.concat([pair_label_means.rename("mean"),
                    _lab_valid_init.groupby(["body_class", "label_class"])["f_label_obj"].std().rename("std"),
                    _lab_valid_init.groupby(["body_class", "label_class"]).size().rename("n")], axis=1))

print("f_cap^{pair} by (body_class, cap_class) (Initial-only, lid excluded):")
display(pd.concat([pair_cap_means.rename("mean"),
                    _cap_valid_init.groupby(["body_class", "cap_class"])["f_cap_obj"].std().rename("std"),
                    _cap_valid_init.groupby(["body_class", "cap_class"]).size().rename("n")], axis=1))

factors_per_object["f_label_pair"] = factors_per_object.set_index(["body_class", "label_class"]).index.map(pair_label_means)
factors_per_object["f_cap_pair"] = factors_per_object.set_index(["body_class", "cap_class"]).index.map(pair_cap_means)

# --- \bar f_r^{pool} (each Initial object weighted equally) and \bar f_r^{bal}
# (each per-type group weighted equally, i.e. mean of the type means) ---
F_LABEL_POOLED   = _lab_valid_init["f_label_obj"].mean()
F_LABEL_BALANCED = label_type_means.mean()
F_CAP_POOLED     = _cap_valid_init["f_cap_obj"].mean()
F_CAP_BALANCED   = cap_type_means.mean()

print()
print(f"f_label^pool: {F_LABEL_POOLED:.4f}   f_label^bal: {F_LABEL_BALANCED:.4f}")
print(f"f_cap^pool:   {F_CAP_POOLED:.4f}   f_cap^bal:   {F_CAP_BALANCED:.4f}")

factors_per_object.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_thickness_factors_per_object.csv", index=False)
print()
print(f"Saved: {RESULTS_DIR / f'{PROJECT_NAME}_thickness_factors_per_object.csv'}")


In [ ]:
# === Cell 14: Apply Correction-Factor Variants ===
# Six variants (EN 18120 only):
#   original    -- 2D area coverage only, no correction (factor = 1)
#   per_object  -- f_r^{object}: each object's own measured thickness ratio
#   per_type    -- \bar f_r^{type}: label_type / cap_type group mean
#   per_material_pair -- \bar f_r^{pair}: (body_class x label_class) /
#                        (body_class x cap_class) group mean
#   global_pooled   -- \bar f_r^{pool}: one number, mean over all Initial objects equally
#   global_balanced -- \bar f_r^{bal}: one number, mean over per-type GROUPS equally
#
# contribution_r,adj = contribution_r * f_r  (severity bounds unchanged).
# A missing factor (f_r = NaN, e.g. printed labels or no-cap objects) falls back
# to the unadjusted contribution (f_r = 1); see the Thickness Correction Factors cell.

VARIANT_LABEL_COL = {
    "original":          None,
    "per_object":        "f_label_obj",
    "per_type":          "f_label_type",
    "per_material_pair": "f_label_pair",
    "global_pooled":      F_LABEL_POOLED,
    "global_balanced":    F_LABEL_BALANCED,
}
VARIANT_CAP_COL = {
    "original":          None,
    "per_object":        "f_cap_obj",
    "per_type":          "f_cap_type",
    "per_material_pair": "f_cap_pair",
    "global_pooled":      F_CAP_POOLED,
    "global_balanced":    F_CAP_BALANCED,
}
VARIANT_ORDER = ["original", "per_object", "per_type", "per_material_pair",
                  "global_pooled", "global_balanced"]

df_variants = df_perspective.merge(
    factors_per_object[["object_name", "f_label_obj", "f_cap_obj",
                        "f_label_type", "f_cap_type", "f_label_pair", "f_cap_pair"]],
    on="object_name", how="left",
)

for _variant in VARIANT_ORDER:
    _lab_spec = VARIANT_LABEL_COL[_variant]
    _cap_spec = VARIANT_CAP_COL[_variant]

    if _lab_spec is None:
        _f_label = 1.0
    elif isinstance(_lab_spec, str):
        _f_label = df_variants[_lab_spec]
    else:
        _f_label = _lab_spec  # scalar (global variants)

    if _cap_spec is None:
        _f_cap = 1.0
    elif isinstance(_cap_spec, str):
        _f_cap = df_variants[_cap_spec]
    else:
        _f_cap = _cap_spec

    _lab_lo = df_variants["label_contribution_lo"] * _f_label
    _lab_hi = df_variants["label_contribution_hi"] * _f_label
    _cap_lo = df_variants["cap_contribution_lo"] * _f_cap
    _cap_hi = df_variants["cap_contribution_hi"] * _f_cap

    # Fall back to the unadjusted contribution wherever the factor is NaN.
    _lab_lo = _lab_lo.fillna(df_variants["label_contribution_lo"]) if isinstance(_lab_lo, pd.Series) else _lab_lo
    _lab_hi = _lab_hi.fillna(df_variants["label_contribution_hi"]) if isinstance(_lab_hi, pd.Series) else _lab_hi
    _cap_lo = _cap_lo.fillna(df_variants["cap_contribution_lo"]) if isinstance(_cap_lo, pd.Series) else _cap_lo
    _cap_hi = _cap_hi.fillna(df_variants["cap_contribution_hi"]) if isinstance(_cap_hi, pd.Series) else _cap_hi

    df_variants[f"I_lo_{_variant}"] = _lab_lo + _cap_lo
    df_variants[f"I_hi_{_variant}"] = _lab_hi + _cap_hi

df_variants.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_gt_index_variants_per_perspective.csv", index=False)
print(f"Saved: {RESULTS_DIR / f'{PROJECT_NAME}_gt_index_variants_per_perspective.csv'}")
print(f"{len(df_variants)} rows, variants: {VARIANT_ORDER}")


In [ ]:
# === Cell 15: Region Size Reference Table ===
# One row per object; label/cap area in mm^2 as "mean +/- std" over the object's
# GT perspectives. Reference only; not used by any later calculation.

_resolvable = df_perspective[df_perspective["index_resolvable"]].copy()


def _mean_std_str(s: pd.Series, present: pd.Series) -> str:
    """Format mean +/- std of s over rows where present is True."""
    vals = s[present]
    if len(vals) == 0:
        return "-"
    if len(vals) == 1:
        return f"{vals.iloc[0]:.1f}"
    return f"{vals.mean():.1f} \u00b1 {vals.std(ddof=0):.1f}"


_rows = []
for _obj, _grp in _resolvable.groupby("object_name"):
    _rows.append({
        "object_name": _obj,
        "body_material": _grp["body_material"].iloc[0],
        "n_perspectives": len(_grp),
        "label_area_mm2 (mean \u00b1 std)": _mean_std_str(_grp["label_area_mm2"], _grp["label_present"]),
        "n_perspectives_with_label": int(_grp["label_present"].sum()),
        "cap_area_mm2 (mean \u00b1 std)": _mean_std_str(_grp["cap_area_mm2"], _grp["cap_present"]),
        "n_perspectives_with_cap": int(_grp["cap_present"].sum()),
    })

region_size_table = pd.DataFrame(_rows).sort_values("object_name").reset_index(drop=True)
display(region_size_table)

region_size_table.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_region_size_per_object.csv", index=False)
print(f"Saved: {RESULTS_DIR / f'{PROJECT_NAME}_region_size_per_object.csv'}")


In [ ]:
# === Cell 16: Index by Body Material Plots ===
# Per-perspective [I_lo, I_hi] ranges as jittered scatter with an I_hi boxplot
# overlay, grouped by body material; one figure per correction-factor variant.

MATERIAL_COLORS = {m: plt.cm.tab10(i) for i, m in enumerate(MATERIAL_ORDER)}

_FW = 16 / 2.54
plt.rcParams["font.size"]       = 10
plt.rcParams["axes.titlesize"]  = 11
plt.rcParams["axes.labelsize"]  = 10
plt.rcParams["legend.fontsize"] = 9
plt.rcParams["savefig.dpi"]     = 150

VARIANT_TITLE = {
    "original":          "2D Area Coverage Only",
    "per_object":        "Per-Object Thickness Factor",
    "per_type":          "Per-Type Thickness Factor",
    "per_material_pair": "Per-Material-Pair Thickness Factor",
    "global_pooled":     "Global Pooled Thickness Factor",
    "global_balanced":   "Global Balanced Thickness Factor",
}


def plot_index_by_material(df, i_lo_col, i_hi_col, title, save_suffix):
    """Plot and save the [I_lo, I_hi] scatter + I_hi boxplot per body material."""
    _df = df[df["body_material"].isin(MATERIAL_ORDER)].copy()
    _df = _df[_df[i_lo_col].notna() & _df[i_hi_col].notna()]
    print(f"Plotting {len(_df)} resolvable perspectives across "
          f"{_df['object_name'].nunique()} objects  -- {title}")

    fig, ax = plt.subplots(figsize=(_FW, _FW * 0.62))
    _rng = np.random.default_rng(RANDOM_SEED)
    _positions = {m: i for i, m in enumerate(MATERIAL_ORDER)}
    _JITTER_WIDTH = 0.28

    _box_data = [_df.loc[_df["body_material"] == m, i_hi_col].to_numpy() for m in MATERIAL_ORDER]
    _bp = ax.boxplot(
        _box_data, positions=list(_positions.values()), widths=0.5,
        patch_artist=True, showfliers=False, showmeans=True, meanline=True,
        medianprops=dict(color="black", lw=1.3, linestyle="-"),
        meanprops=dict(color="black", lw=1.1, linestyle="--"),
        zorder=2,
    )
    from matplotlib.colors import to_rgba
    for _patch, _m in zip(_bp["boxes"], MATERIAL_ORDER):
        _face_rgba = to_rgba(MATERIAL_COLORS[_m], alpha=0.18)
        _patch.set_facecolor(_face_rgba)
        _patch.set_edgecolor("black")
        _patch.set_linewidth(0.8)

    for _m in MATERIAL_ORDER:
        _sub = _df[_df["body_material"] == _m]
        if _sub.empty:
            continue
        _x0 = _positions[_m]
        _jitter = _rng.uniform(-_JITTER_WIDTH, _JITTER_WIDTH, size=len(_sub))
        _x = _x0 + _jitter
        _color = MATERIAL_COLORS[_m]

        _lo = _sub[i_lo_col].to_numpy()
        _hi = _sub[i_hi_col].to_numpy()
        _is_point = np.isclose(_lo, _hi)

        ax.vlines(_x[~_is_point], _lo[~_is_point], _hi[~_is_point],
                   color=_color, alpha=0.55, linewidth=1.1, zorder=3)
        ax.scatter(_x[~_is_point], _lo[~_is_point], color=_color, s=8, alpha=0.7, zorder=4)
        ax.scatter(_x[~_is_point], _hi[~_is_point], color=_color, s=8, alpha=0.7, zorder=4)
        ax.scatter(_x[_is_point], _lo[_is_point], color=_color, s=10, alpha=0.45,
                   marker="o", edgecolor="none", zorder=3)

        if SHOW_VETO_MARKERS:
            _veto_mask = _sub["veto_strict"].to_numpy() == 1
            if _veto_mask.any():
                ax.scatter(_x[_veto_mask], _hi[_veto_mask], facecolor="none", edgecolor="black",
                           s=45, linewidth=1.1, marker="o", zorder=5)

    ax.set_xticks(list(_positions.values()))
    ax.set_xticklabels(MATERIAL_ORDER)
    ax.set_xlim(-0.6, len(MATERIAL_ORDER) - 0.4)
    _ymax = _df[i_hi_col].max() if len(_df) else 1.0
    ax.set_ylim(0, _ymax * 1.18 if _ymax > 0 else 1.0)
    ax.set_ylabel(r"Impurity Index $I$")
    ax.set_xlabel("Body Material")
    ax.set_title(title)

    for i in range(len(MATERIAL_ORDER)):
        if i % 2 == 0:
            ax.axvspan(i - 0.5, i + 0.5, color="grey", alpha=0.06, zorder=0)

    _legend_elems = [
        Line2D([0], [0], marker="o", color="none", markerfacecolor="grey",
               markeredgecolor="none", markersize=6, alpha=0.7, label="Perspective"),
        Line2D([0], [0], color="black", lw=1.3, linestyle="-", label=r"Median ($I_{hi}$)"),
        Line2D([0], [0], color="black", lw=1.1, linestyle="--", label=r"Mean ($I_{hi}$)"),
    ]
    if SHOW_VETO_MARKERS:
        _legend_elems.insert(1, Line2D([0], [0], marker="o", color="none", markerfacecolor="none",
                                        markeredgecolor="black", markersize=7, label="veto_strict = 1"))
    ax.legend(handles=_legend_elems, loc="upper right", framealpha=0.9)

    plt.tight_layout()
    plt.show()

    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    fig_path = RESULTS_DIR / f"{PROJECT_NAME}_index_by_material_{save_suffix}.png"
    fig.savefig(fig_path, dpi=150, bbox_inches="tight")
    print(f"Saved: {fig_path}")
    return fig


for _variant in VARIANT_ORDER:
    plot_index_by_material(
        df_variants, f"I_lo_{_variant}", f"I_hi_{_variant}",
        f"Impurity Index based on {VARIANT_TITLE[_variant]}",
        _variant,
    )


In [ ]:
# === Cell 17: Severity Interval Width Comparison ===
# Raw severity width (severity_hi - severity_lo) per present region on resolvable
# perspectives, EN 18120 vs RecyClass, label and cap pooled. Widths take only the
# values 0, 0.5 and 1, so the x-axis is categorical.
# EN 18120: solid tab:blue; RecyClass: hatched tab:orange with a black outline stroke.

import matplotlib.patheffects as patheffects

_gt = df_perspective[df_perspective["index_resolvable"]].copy()
_gt = _gt[_gt["body_material"].isin(MATERIAL_ORDER)]

_series = [
    ("label", "EN18120",   "label_severity_lo",    "label_severity_hi",    "label_present"),
    ("label", "RecyClass", "label_severity_lo_rc", "label_severity_hi_rc", "label_present"),
    ("cap",   "EN18120",   "cap_severity_lo",       "cap_severity_hi",      "cap_present"),
    ("cap",   "RecyClass", "cap_severity_lo_rc",    "cap_severity_hi_rc",   "cap_present"),
]
WIDTH_CATEGORIES = [0.0, 0.5, 1.0]

_pooled = {"EN18120": [], "RecyClass": []}
for _component, _table, _lo_col, _hi_col, _present_col in _series:
    _sub = _gt[_gt[_present_col]]
    _w = (_sub[_hi_col] - _sub[_lo_col]).dropna()
    _pooled[_table].append(_w)

_counts = {}
for _table in ["EN18120", "RecyClass"]:
    _w_all = pd.concat(_pooled[_table])
    _counts[_table] = _w_all.value_counts().reindex(WIDTH_CATEGORIES, fill_value=0)
    print(f"{_table:9s}: n={len(_w_all):3d}  " +
          "  ".join(f"{cat}:{int(_counts[_table][cat])}" for cat in WIDTH_CATEGORIES))

_unexpected = set(pd.concat(_pooled["EN18120"] + _pooled["RecyClass"]).unique()) - set(WIDTH_CATEGORIES)
if _unexpected:
    raise ValueError(f"Severity width value(s) outside {{0, 0.5, 1}} found: {_unexpected} -- "
                      "check severity_lookup_EN18120.csv / severity_lookup_RecyClass.csv for a new tier.")

_FW = 16 / 2.54
plt.rcParams["font.size"]      = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["legend.fontsize"] = 10

fig, ax = plt.subplots(figsize=(_FW, _FW * 0.62))

_x = np.arange(len(WIDTH_CATEGORIES))
_bar_width = 0.32
TABLE_COLOR = {"EN18120": "tab:blue", "RecyClass": "tab:orange"}

_bars_en = ax.bar(_x - _bar_width / 2, _counts["EN18120"].to_numpy(), width=_bar_width * 0.92,
                   color=TABLE_COLOR["EN18120"], alpha=0.85, edgecolor="black", linewidth=0.3, label="EN18120")

_bars_rc = ax.bar(_x + _bar_width / 2, _counts["RecyClass"].to_numpy(), width=_bar_width * 0.92,
                   color=TABLE_COLOR["RecyClass"], alpha=0.75, hatch="///", edgecolor="white", linewidth=0.5, label="RecyClass")
for _patch in _bars_rc:
    _patch.set_path_effects([patheffects.withStroke(linewidth=1.2, foreground="black")])

ax.set_xticks(_x)
ax.set_xticklabels([f"{v:.1f}" for v in WIDTH_CATEGORIES])
ax.set_xlabel(r"Severity Width $(s_{hi} - s_{lo})$")
ax.set_ylabel("Perspective Count")
ax.set_title("Severity Interval Width of EN18120 and RecyClass")
ax.legend(loc="upper right", framealpha=0.9)

for i in range(len(WIDTH_CATEGORIES)):
    if i % 2 == 0:
        ax.axvspan(i - 0.5, i + 0.5, color="#f0f0f0", zorder=0)
ax.set_xlim(-0.5, len(WIDTH_CATEGORIES) - 0.5)

plt.tight_layout()
plt.show()

_fig_path = RESULTS_DIR / f"{PROJECT_NAME}_severity_width_en_vs_recyclass.png"
fig.savefig(_fig_path, dpi=150, bbox_inches="tight")
print(f"Saved: {_fig_path}")


In [ ]:
# === Cell 18: Factor-Level Summary Table ===
# Mean I_lo/I_hi per variant over perspectives resolvable under all variants, plus
# Pearson correlation and mean absolute difference of I_hi vs the per_object variant.

_valid = df_variants.copy()
for _variant in VARIANT_ORDER:
    _valid = _valid[_valid[f"I_lo_{_variant}"].notna() & _valid[f"I_hi_{_variant}"].notna()]

print(f"Resolvable perspectives across ALL variants: {len(_valid)}")

_summary_rows = []
for _variant in VARIANT_ORDER:
    _row = {
        "variant": _variant,
        "I_lo_mean": _valid[f"I_lo_{_variant}"].mean(),
        "I_hi_mean": _valid[f"I_hi_{_variant}"].mean(),
    }
    if _variant == "per_object":
        _row["corr_I_hi_vs_per_object"] = 1.0
        _row["mean_abs_diff_I_hi_vs_per_object"] = 0.0
    else:
        _row["corr_I_hi_vs_per_object"] = _valid[f"I_hi_{_variant}"].corr(_valid["I_hi_per_object"])
        _row["mean_abs_diff_I_hi_vs_per_object"] = (
            _valid[f"I_hi_{_variant}"] - _valid["I_hi_per_object"]
        ).abs().mean()
    _summary_rows.append(_row)

factor_level_summary = pd.DataFrame(_summary_rows).set_index("variant").loc[VARIANT_ORDER]
display(factor_level_summary)

factor_level_summary.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_factor_level_summary.csv")
print(f"Saved: {RESULTS_DIR / f'{PROJECT_NAME}_factor_level_summary.csv'}")


In [ ]:
# === Cell 19: Factor-Level Trade-off Plot ===
# Two panels sharing the variant x-axis:
#   top    -- I_hi boxplot per variant (perspectives resolvable under all variants).
#   bottom -- corr(I_hi_variant, I_hi_per_object) per variant.

_valid17 = df_variants.copy()
for _variant in VARIANT_ORDER:
    _valid17 = _valid17[_valid17[f"I_lo_{_variant}"].notna() & _valid17[f"I_hi_{_variant}"].notna()]

# Short x-axis labels for the variants.
VARIANT_SHORT = {
    "original":          "Original",
    "per_object":        "Per-Object",
    "per_type":          "Per-Type",
    "per_material_pair": "Per-Material-Pair",
    "global_pooled":     "Global Pooled",
    "global_balanced":   "Global Balanced",
}
_xtick_labels = [VARIANT_SHORT[v] for v in VARIANT_ORDER]

_FW = 16 / 2.54
fig, axes = plt.subplots(2, 1, figsize=(_FW, _FW * 1.1), sharex=True)

# --- Top: I_hi boxplot per variant ---
_box_data = [_valid17[f"I_hi_{v}"].to_numpy() for v in VARIANT_ORDER]
_bp = axes[0].boxplot(
    _box_data, positions=range(len(VARIANT_ORDER)), widths=0.6,
    patch_artist=True, showfliers=False, showmeans=True, meanline=True,
    medianprops=dict(color="black", lw=1.3, linestyle="-"),
    meanprops=dict(color="black", lw=1.1, linestyle="--"),
    zorder=2,
)
for _patch in _bp["boxes"]:
    _patch.set_facecolor("#bfbfbf")
    _patch.set_edgecolor("black")
    _patch.set_linewidth(0.8)
axes[0].set_ylabel(r"$I_{hi}$")
axes[0].set_title(r"$I_{hi}$ Distribution")

# --- Bottom: correlation vs per_object per variant ---
_corr_vals = [factor_level_summary.loc[v, "corr_I_hi_vs_per_object"] for v in VARIANT_ORDER]
axes[1].plot(range(len(VARIANT_ORDER)), _corr_vals, marker="o", color="black",
             linewidth=1.3, zorder=3)
axes[1].axhline(1.0, color="grey", linestyle=":", linewidth=0.8, zorder=1)
axes[1].set_xticks(range(len(VARIANT_ORDER)))
axes[1].set_xticklabels(_xtick_labels, rotation=20, ha="right", fontsize=9)
axes[1].set_ylabel(r"$\mathrm{corr}(I_{hi},\ I_{hi}^{\mathrm{object}})$")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Object-Level Fidelity")

# Alternating grey/white bands on the categorical axis, both panels.
for _ax in axes:
    for _i in range(len(VARIANT_ORDER)):
        if _i % 2 == 0:
            _ax.axvspan(_i - 0.5, _i + 0.5, color="grey", alpha=0.06, zorder=0)
    _ax.set_xlim(-0.5, len(VARIANT_ORDER) - 0.5)

fig.suptitle(r"Thickness Correction $f_r$ Aggregation Level Trade-off", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

_fig_path = RESULTS_DIR / f"{PROJECT_NAME}_factor_level_tradeoff.png"
fig.savefig(_fig_path, dpi=150, bbox_inches="tight")
print(f"Saved: {_fig_path}")


In [ ]:
# === Cell 20: Per-Object Perspective Spread ===
# Within-object spread of I_hi across perspectives (std, range), using the
# "original" variant; per-object thickness factors are constant per object.

_resolvable = df_variants[df_variants["I_hi_original"].notna()].copy()

_per_object_spread = _resolvable.groupby("object_name").agg(
    body_material=("body_material", "first"),
    n_perspectives=("I_hi_original", "size"),
    I_hi_mean=("I_hi_original", "mean"),
    I_hi_std=("I_hi_original", lambda s: s.std(ddof=0)),
    I_hi_range=("I_hi_original", lambda s: s.max() - s.min()),
).reset_index()

print("Per-object I_hi spread across perspectives (original / 2D-area weighting):")
display(_per_object_spread.sort_values("I_hi_std", ascending=False).head(10))

print()
print("Mean within-object spread, by body material:")
_material_spread = _per_object_spread.groupby("body_material").agg(
    n_objects=("object_name", "nunique"),
    mean_I_hi_std=("I_hi_std", "mean"),
    mean_I_hi_range=("I_hi_range", "mean"),
).loc[MATERIAL_ORDER]
display(_material_spread)

_per_object_spread.to_csv(RESULTS_DIR / f"{PROJECT_NAME}_per_object_perspective_spread.csv", index=False)
print(f"Saved: {RESULTS_DIR / f'{PROJECT_NAME}_per_object_perspective_spread.csv'}")


In [ ]:
# === Cell 21: Evaluation-Tier Segmentation Setup ===
# From here on the `yolo-train` kernel is required: evaluation-tier YOLO inference
# on the held-out test objects. Only the coverage ratio w_r is compared, predicted
# vs GT, as a signed difference (relative error is unstable for w_r,gt near 0):
#   delta_w_r = w_r,pred - w_r,gt      r in {label, cap}
# Uses _check from the Environment and Path Checks cell.

import torch
import ultralytics
from ultralytics import YOLO

print(f"torch:       {torch.__version__}")
print(f"ultralytics: {ultralytics.__version__}")

EVAL_DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    print(f"CUDA:        {torch.cuda.get_device_name(0)}")
else:
    print("CUDA:        not available -- inference will run on CPU.")

# --- Evaluation-tier model and object split ---
# Model trained on the CV pool only; test objects are held out.
SEG_MODEL_EVAL = Path(
    f"{KI_SORT_ROOT}/KI_Sort_Project/02_Image_Segmentation/"
    "20260529_Yolo_Final_Training_yolov8n/runs/"
    "20260529_Yolo_Final_Training_yolov8n_20260529_Yolo_Final_Training_yolov8n/"
    "weights/best.pt"
)
OBJECT_SPLIT_CSV = Path(f"{KI_SORT_ROOT}/KI_Sort_Project/01_LS_Export/20260518_Export/object_split.csv")

EVAL_CONF_THRESHOLD = 0.50
EVAL_IOU_THRESHOLD  = 0.50
EVAL_IMAGE_SIZE     = 1080
EVAL_MAX_DET        = 300

_ok = True
_ok &= _check("SEG_MODEL_EVAL",    SEG_MODEL_EVAL,    "file")
_ok &= _check("OBJECT_SPLIT_CSV",  OBJECT_SPLIT_CSV,  "file")
if not _ok:
    raise FileNotFoundError("Evaluation-tier asset(s) missing -- see [MISSING] lines above.")
print("All evaluation-tier paths OK.")


In [ ]:
# === Cell 22: Evaluation-Tier Predicted Coverage ===

object_split = {}
_sp = pd.read_csv(OBJECT_SPLIT_CSV)
_sp.columns = [c.strip() for c in _sp.columns]
for _, _r in _sp.iterrows():
    _mat = str(_r["material"]).strip().upper()
    _oid = str(_r["object_id"]).strip()
    _digits = re.sub(r"\D", "", _oid)
    if not _digits:
        continue
    object_split[f"{_mat}_obj_{_digits.zfill(2)}"] = str(_r["split"]).strip().lower()

_test_objects = sorted(k for k, v in object_split.items() if v == "test")
print(f"Object split rows: {len(object_split)}  ({len(_test_objects)} test)")
print(f"Test objects: {_test_objects}")


def lookup_split(object_name):
    """Split name for an object, or "unlisted"."""
    return object_split.get(object_name, "unlisted")


print()
print(f"Loading evaluation-tier YOLO: {SEG_MODEL_EVAL.name}")
yolo_model_eval = YOLO(str(SEG_MODEL_EVAL))


def segment_image(image_path, h, w):
    """Run evaluation-tier YOLO on one image; returns the same instance dicts as
    parse_label_file, so compute_coverage applies unchanged."""
    res = yolo_model_eval.predict(
        source=str(image_path), conf=EVAL_CONF_THRESHOLD, iou=EVAL_IOU_THRESHOLD,
        imgsz=EVAL_IMAGE_SIZE, max_det=EVAL_MAX_DET, device=EVAL_DEVICE, verbose=False,
    )[0]
    instances = []
    if res.masks is None:
        return instances
    for poly, cls_id in zip(res.masks.xyn, res.boxes.cls.tolist()):
        if poly is None or len(poly) < 3:
            continue
        instances.append({
            "class_id":     int(cls_id),
            "polygon_norm": [(float(x), float(y)) for x, y in poly],
        })
    return instances


# --- Predicted coverage, one row per test-object perspective ---
_pred_rows = []
_test_work = []
for _obj in _test_objects:
    for _per_id, _img_path in image_index.get(_obj, {}).items():
        _test_work.append((_obj, _per_id, _img_path.stem, _img_path))
_test_work.sort(key=lambda t: (t[0], t[1]))

print()
print(f"Running evaluation-tier inference on {len(_test_work)} perspective(s) "
      f"across {len(_test_objects)} test object(s) ...")

_t0 = datetime.now()
for _i, (_obj, _per_id, _stem, _img_path) in enumerate(_test_work, start=1):
    _h, _w = resolve_image_size(_img_path)
    _instances = segment_image(_img_path, _h, _w)
    _coverage = compute_coverage(_instances, _h, _w)
    _pred_rows.append({
        "stem": _stem, "object_name": _obj, "perspective_id": _per_id,
        **_coverage,
    })
    if _i % 20 == 0 or _i == len(_test_work):
        print(f"  {_i}/{len(_test_work)}")

df_pred_coverage = pd.DataFrame(_pred_rows)
_elapsed = (datetime.now() - _t0).total_seconds()
_n_undefined = int(df_pred_coverage["undefined"].sum())
print(f"Done in {_elapsed:.1f}s -- {len(df_pred_coverage)} rows")
print(f"undefined (no detections at all): {_n_undefined} / {len(df_pred_coverage)}")


In [ ]:
# === Cell 23: Predicted vs GT Coverage Delta ===
# Merges predicted and GT coverage per test perspective and computes delta_w_r.

_gt_test = df_perspective[df_perspective["object_name"].isin(_test_objects)][
    ["stem", "object_name", "perspective_id", "w_label", "w_cap",
     "label_present", "cap_present", "undefined"]
].copy()

df_eval_compare = df_pred_coverage[
    ["stem", "object_name", "perspective_id", "w_label", "w_cap",
     "label_present", "cap_present", "undefined"]
].merge(_gt_test, on=["stem", "object_name", "perspective_id"], how="inner",
        suffixes=("_pred", "_gt"))

print(f"Matched {len(df_eval_compare)} perspective(s) "
      f"(predicted rows: {len(df_pred_coverage)}, GT test rows: {len(_gt_test)})")

_pred_miss = int((df_eval_compare["undefined_pred"] & ~df_eval_compare["undefined_gt"]).sum())
if _pred_miss:
    print(f"NOTE: {_pred_miss} perspective(s) where the model detected NOTHING at all "
          f"(w_label_pred = w_cap_pred = NaN) while GT has a real segmentation -- "
          "these are total detection misses, a different failure mode from a boundary/"
          "sizing error, and are excluded from delta_w_r below (NaN propagates).")

df_eval_compare["delta_w_label"] = df_eval_compare["w_label_pred"] - df_eval_compare["w_label_gt"]
df_eval_compare["delta_w_cap"]   = df_eval_compare["w_cap_pred"]   - df_eval_compare["w_cap_gt"]

_eval_summary_rows = []
for _region in ["label", "cap"]:
    _d = df_eval_compare[f"delta_w_{_region}"].dropna()
    _row = {
        "region": _region, "n": len(_d),
        "mean": _d.mean(), "std": _d.std(ddof=0), "median": _d.median(),
        "min": _d.min(), "max": _d.max(),
    }
    _eval_summary_rows.append(_row)
    print(f"delta_w_{_region}: n={_row['n']}  mean={_row['mean']:+.4f}  std={_row['std']:.4f}  "
          f"median={_row['median']:+.4f}  min={_row['min']:+.4f}  max={_row['max']:+.4f}")

eval_coverage_summary = pd.DataFrame(_eval_summary_rows).set_index("region")

_out_path = RESULTS_DIR / f"{PROJECT_NAME}_eval_coverage_comparison.csv"
df_eval_compare.to_csv(_out_path, index=False)
print(f"Saved: {_out_path}")

_summary_out_path = RESULTS_DIR / f"{PROJECT_NAME}_eval_coverage_summary.csv"
eval_coverage_summary.to_csv(_summary_out_path)
print(f"Saved: {_summary_out_path}")


In [ ]:
# === Cell 24: Predicted vs GT Coverage Plot ===
# Predicted vs GT w_r, one point per test-object perspective, with a 1:1 line.
# label = tab:blue, cap = tab:green.

REGION_COLOR = {"label": "tab:blue", "cap": "tab:green"}

_FW_SQ = 8 / 2.54   # 8 cm square
fig, ax = plt.subplots(figsize=(_FW_SQ, _FW_SQ))

for _region in ["label", "cap"]:
    _sub = df_eval_compare[[f"w_{_region}_gt", f"w_{_region}_pred"]].dropna()
    ax.scatter(_sub[f"w_{_region}_gt"], _sub[f"w_{_region}_pred"],
               color=REGION_COLOR[_region], alpha=0.7, s=28,
               edgecolor="black", linewidth=0.4, label=_region, zorder=3)

# 1:1 reference line (no legend entry).
_lims = [0, 1]
ax.plot(_lims, _lims, color="black", linestyle="--", linewidth=1.0, zorder=2)

ax.set_xlim(_lims)
ax.set_ylim(_lims)
ax.set_aspect("equal")
ax.set_xlabel(r"$w_r$ (Ground Truth)")
ax.set_ylabel(r"$w_r$ (Predicted)")
ax.legend(loc="lower right", framealpha=0.9, fontsize=8)

plt.tight_layout()
plt.show()

_fig_path = RESULTS_DIR / f"{PROJECT_NAME}_eval_coverage_pred_vs_gt.png"
fig.savefig(_fig_path, dpi=150, bbox_inches="tight")
print(f"Saved: {_fig_path}")


In [ ]:
# === Cell 25: Verification of §5.5 Statistics ===
# Recomputes the derived numbers of thesis §5.5 from this notebook's exported
# CSVs. Prints only; writes no files.

import pandas as pd
import numpy as np

df_persp   = pd.read_csv(OUTPUT_DIR / "20260821_Impurity_Index_Computation_gt_index_per_perspective.csv")
df_variant = pd.read_csv(OUTPUT_DIR / "20260821_Impurity_Index_Computation_gt_index_variants_per_perspective.csv")
df_thick   = pd.read_csv(OUTPUT_DIR / "20260821_Impurity_Index_Computation_thickness_factors_per_object.csv")
df_cov     = pd.read_csv(OUTPUT_DIR / "20260821_Impurity_Index_Computation_eval_coverage_comparison.csv")

pd.set_option("display.float_format", lambda x: f"{x:.4f}")


# --- §5.5 opening: resolvable perspectives ---
print("=" * 70)
print("§5.5 opening — index resolvability")
print("=" * 70)

n_total = len(df_persp)
n_resolvable = df_persp.index_resolvable.sum()
print(f"resolvable perspectives: {n_resolvable} / {n_total} "
      f"({100 * n_resolvable / n_total:.1f}%)")

unresolved = df_persp[~df_persp.index_resolvable]
print(f"unresolved perspectives: {len(unresolved)}, "
      f"from {unresolved.object_name.nunique()} objects: "
      f"{sorted(unresolved.object_name.unique())}")
print(f"all unresolved rows have body_present == False: "
      f"{(unresolved.body_present == False).all()}")


# --- §5.5.1: EN 18120 vs RecyClass severity width ---
print("\n" + "=" * 70)
print("§5.5.1 — severity interval width, EN18120 vs. RecyClass")
print("=" * 70)

resolvable = df_persp[df_persp.index_resolvable]

def severity_widths(df, comp):
    """Rows with resolved severity for comp, plus EN 18120 and RecyClass widths."""
    ok = df[df[f"{comp}_severity_status"] == "ok"]
    en = ok[f"{comp}_severity_hi"] - ok[f"{comp}_severity_lo"]
    rc = ok[f"{comp}_severity_hi_rc"] - ok[f"{comp}_severity_lo_rc"]
    return ok, en, rc

label_ok, label_en, label_rc = severity_widths(resolvable, "label")
cap_ok, cap_en, cap_rc       = severity_widths(resolvable, "cap")

en_pooled = pd.concat([label_en, cap_en])
rc_pooled = pd.concat([label_rc, cap_rc])
n_pooled = len(en_pooled)
print(f"pooled component instances: {n_pooled} "
      f"({len(label_ok)} label + {len(cap_ok)} cap)")

for tier in [0.0, 0.5, 1.0]:
    n_en = (en_pooled == tier).sum()
    n_rc = (rc_pooled == tier).sum()
    print(f"  width {tier}: EN18120 {n_en}/{n_pooled} ({100*n_en/n_pooled:.1f}%)"
          f"   RecyClass {n_rc}/{n_pooled} ({100*n_rc/n_pooled:.1f}%)")

print(f"\ncap-only, EN18120 width==0: {(cap_en == 0).sum()}/{len(cap_ok)} "
      f"({100*(cap_en==0).mean():.1f}%)")
print(f"cap-only, RecyClass width==0: {(cap_rc == 0).sum()}/{len(cap_ok)}, "
      f"width==0.5: {(cap_rc == 0.5).sum()}/{len(cap_ok)}")


# --- §5.5.1: EN 18120 vs RecyClass divergent severities ---
print("\n" + "=" * 70)
print("§5.5.1 — EN18120 / RecyClass divergent cells")
print("=" * 70)

div_label = (label_ok.label_severity_lo != label_ok.label_severity_lo_rc) | \
            (label_ok.label_severity_hi != label_ok.label_severity_hi_rc)
div_cap   = (cap_ok.cap_severity_lo != cap_ok.cap_severity_lo_rc) | \
            (cap_ok.cap_severity_hi != cap_ok.cap_severity_hi_rc)

n_div = div_label.sum() + div_cap.sum()
print(f"diverging component instances: {n_div}/{n_pooled} "
      f"({100 * n_div / n_pooled:.1f}%)")

print("\nby material combination (label):")
print(label_ok[div_label].groupby(["body_material", "label_material"]).size())
print("\nby material combination (cap):")
print(cap_ok[div_cap].groupby(["body_material", "cap_material"]).size())


# --- §5.5.2: thickness ratio medians (f_r) ---
print("\n" + "=" * 70)
print("§5.5.2 — per-object thickness correction factor f_r")
print("=" * 70)

n_obj = len(df_thick)
for r in ["label", "cap"]:
    col = f"f_{r}_obj"
    n = df_thick[col].notna().sum()
    med = df_thick[col].median()
    print(f"  f_{r},obj: median {med:.3f} over {n}/{n_obj} objects")


# --- §5.5.2: factor-level summary (tab:res-index-factors) ---
print("\n" + "=" * 70)
print("§5.5.2 — index by thickness-correction level")
print("=" * 70)

variants = ["original", "per_object", "per_type",
            "per_material_pair", "global_pooled", "global_balanced"]

r_persp = df_variant[df_variant.index_resolvable]
ref = r_persp["I_hi_per_object"]

rows = []
for v in variants:
    hi = r_persp[f"I_hi_{v}"]
    lo = r_persp[f"I_lo_{v}"]
    corr = hi.corr(ref)
    mad = (hi - ref).abs().mean()
    rows.append([v, lo.mean(), hi.mean(), corr, mad])

summary = pd.DataFrame(rows, columns=["variant", "I_lo_mean", "I_hi_mean", "corr_vs_per_object", "mad_vs_per_object"])
print(summary.round(3).to_string(index=False))

corrected = summary[summary.variant != "original"]
print(f"\nI_hi mean span across corrected variants: "
      f"{corrected.I_hi_mean.max() - corrected.I_hi_mean.min():.3f} "
      f"(min {corrected.I_hi_mean.min():.3f}, max {corrected.I_hi_mean.max():.3f})")

coarser = summary[summary.variant.isin(["per_material_pair", "global_pooled", "global_balanced"])]
print(f"coarser-level correlation range: "
      f"{coarser.corr_vs_per_object.min():.3f}–{coarser.corr_vs_per_object.max():.3f}")


# --- §5.5.4: index distribution, per_type correction ---
print("\n" + "=" * 70)
print("§5.5.4 — index distribution, per_type correction")
print("=" * 70)

hi = r_persp["I_hi_per_type"]
lo = r_persp["I_lo_per_type"]
width = hi - lo

print(f"overall: mean {hi.mean():.3f}, median {hi.median():.3f}")
print(f"zero share (I_hi == 0): {(hi == 0).mean():.3f}")
print(f"admissibility interval width: mean {width.mean():.3f}, median {width.median():.3f}")

print("\nper-material mean I_hi (per_type):")
print(r_persp.groupby("body_material")["I_hi_per_type"].mean().round(3))


# --- §5.5.5: evaluation-tier coverage error ---
print("\n" + "=" * 70)
print("§5.5.5 — evaluation-tier coverage error")
print("=" * 70)

for region in ["label", "cap"]:
    d = df_cov[f"delta_w_{region}"]
    print(f"{region}: n={d.notna().sum()}, mean {d.mean():.3f}, "
          f"std(ddof=0) {d.std(ddof=0):.3f}, median {d.median():.3f}")

print()
for region in ["label", "cap"]:
    gt = df_cov[f"{region}_present_gt"]
    pr = df_cov[f"{region}_present_pred"]
    tp = ((gt) & (pr)).sum()
    fn = ((gt) & (~pr)).sum()
    fp = ((~gt) & (pr)).sum()
    tn = ((~gt) & (~pr)).sum()
    print(f"{region} presence — TP {tp}, FN {fn}, FP {fp}, TN {tn}")

    agree = df_cov[gt & pr]
    d = agree[f"delta_w_{region}"]
    print(f"  agreement-only (n={len(agree)}): mean {d.mean():.3f}, "
          f"std(ddof=0) {d.std(ddof=0):.3f}, median {d.median():.3f}")

print("\nworst-case rescaling example (label miss inflates cap weight):")
example = df_cov[df_cov.object_name == "PET_obj_04"][
    ["perspective_id", "w_label_gt", "w_cap_gt", "w_cap_pred", "label_present_pred"]
]
print(example.to_string(index=False))